In [ ]:
import numpy as np
from PIL import Image

def load_mask_from_png(image_path, mask_num):
    # 读取PNG图像
    img = Image.open(image_path)
    
    # 转换为numpy数组
    mask = np.array(img)
    return mask == mask_num

# 示例调用
mask = load_mask_from_png('/mnt/petrelfs/yanziang/RefVOS/train/Annotations/003234408d/00000.png', 1)
print(mask.shape)  # 输出mask的形状

In [ ]:
from petrel_client.client import Client
c = Client()
# for tt in c.list('shdd:s3://davis/DAVIS/Annotations/480p/bear'):
#     print(tt)
eval(c.get('shdd:s3://davis/Annotations_json/bear/1.json').decode())["gt_mask"]

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm
from petrel_client.client import Client
c = Client()
def numpy_array_to_video(numpy_array, video_out_path, fps=30):
    video_height, video_width, channels = numpy_array.shape[1], numpy_array.shape[2], numpy_array.shape[3]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  
    video_writer = cv2.VideoWriter(video_out_path, fourcc, fps, (video_width, video_height))
    for frame in numpy_array:
        video_writer.write(frame)
    video_writer.release()

for dd in tqdm(c.list('shdd:s3://davis/DAVIS/JPEGImages/480p')):
    video = None
    for img in c.list(f'shdd:s3://davis/DAVIS/JPEGImages/480p/{dd}'):
        img_url = f'shdd:s3://davis/DAVIS/JPEGImages/480p/{dd}{img}'
        img_bytes = c.get(img_url)
        assert(img_bytes is not None)
        img_mem_view = memoryview(img_bytes)
        img_array = np.frombuffer(img_mem_view, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        if video is None:
            video = np.expand_dims(img, axis=0)
        else:
            img = np.expand_dims(img, axis=0)
            video = np.append(video, img, axis=0)
    numpy_array_to_video(video, 'output_video.mp4')
    
    with open('output_video.mp4', 'rb') as file_data:
        c.put(f'shdd:s3://davis/videos/{dd[:-1]}.mp4',file_data)
    
    

In [ ]:
# [{"exp": "front elephant walking to backwards", "mask": "pnorm2:s3://mevis/masks_val_u/0.json", "video_path": "pnorm2:s3://mevis/videos/d56a6ec78cfa.mp4"}
ret = []
i = 0
import pycocotools.mask as mask_util
import os
for key in c.list('shdd:s3://davis/videos/'):
    video_path = 'shdd:s3://davis/videos/' + key 
    exp = key.split('.mp4')[0]
    print(video_path, exp)
    anno_path = '/mnt/petrelfs/yanziang/davis/DAVIS/Annotations/480p/' + exp
    anno_id = 1
    anno_json_path = 'shdd:s3://davis/Annotations_json/' + exp + '/' + str(anno_id) + '.json'
    json_anno = []
    masks = []
    anno_files = os.listdir(anno_path)
    anno_files.sort() 
    for anno_file in anno_files:
        mask = load_mask_from_png(anno_path + '/' + anno_file, anno_id)
        masks.append(mask)
    masks = np.array(masks)
    masks = masks.transpose(1,2,0) 
#     # 
    masks = str(mask_util.encode(np.asfortranarray(masks)))
    masks = {'gt_mask': masks}
#     # print(masks)
    c.put( anno_json_path, str(masks) )
    ret.append({'exp':exp, 'mask':anno_json_path, 'video_path': video_path})
    # i = i+1
#     # print(i)
    

In [ ]:
with open('davis17.json','w') as f:
    json.dump(ret, f)

In [ ]:
# import json
# with open('/mnt/petrelfs/yanziang/RefVOS/rvos/meta_expressions/train/meta_expressions.json', 'r') as f:
#     data = json.load(f)
# data = data['videos']

import json
with open('/mnt/petrelfs/yanziang/RefVOS/rvos/meta_expressions/valid/meta_expressions.json', 'r') as f:
    data = json.load(f)
data = data['videos']


In [ ]:
tt = c.list('shdd:s3://refvos-youtube/Annotations_json/0062f687f1')
for dd in tt:
    print(dd)

In [ ]:
# [{"exp": "front elephant walking to backwards", "mask": "pnorm2:s3://mevis/masks_val_u/0.json", "video_path": "pnorm2:s3://mevis/videos/d56a6ec78cfa.mp4"}
ret = []
i = 0
import pycocotools.mask as mask_util
import os
for key in data:
    video_path = 'shdd:s3://refvos-youtube/videos/' + key + '.mp4'
    anno_path = '/mnt/petrelfs/yanziang/RefVOS/rvos/train/Annotations/' + key
    for exp_id in data[key]['expressions']:
        exp = data[key]['expressions'][exp_id]['exp']
        anno_id = int(data[key]['expressions'][exp_id]['obj_id'])
        anno_json_path = 'shdd:s3://refvos-youtube/Annotations_json/' + key + '/' + str(anno_id) + '.json'
        json_anno = []
        
        anno_files = os.listdir(anno_path)
        anno_files.sort() 
        masks = []
        for anno_file in anno_files:
            mask = load_mask_from_png(anno_path + '/' + anno_file, anno_id)
            masks.append(mask)
        masks = np.array(masks)
        masks = masks.transpose(1,2,0) 
        # 
        masks = str(mask_util.encode(np.asfortranarray(masks)))
        masks = {'gt_mask': masks}
        # print(masks)
        c.put( anno_json_path, str(masks) )
        ret.append({'exp':exp, 'mask':anno_json_path, 'video_path': video_path})
        i = i+1
        # print(i)
        

In [ ]:
# print(ret)
print(len(ret))
with open('refros_youtube.json','w') as f:
    json.dump(ret, f)

In [ ]:
for tt in c.list('shdd:s3://refvos-youtube/'):
    print(tt)

In [ ]:
path = 'shdd:s3://refvos-youtube/Annotations_json/003234408d/1.json'
gt_masks = eval(c.get(path).decode())['gt_mask']

In [ ]:
import json
with open('/mnt/petrelfs/yanziang/trackingllm/videoagent/davis17.json', 'r') as f:
    data = json.load(f)
    

In [ ]:
import pycocotools.mask as mask_util
fin = []
for dd in data:
    # print(dd)
    path = dd['mask']
    gt_masks = eval(eval(c.get(path).decode())['gt_mask'])
    gt_masks = mask_util.decode(gt_masks) > 0
    
    gt_masks = gt_masks[:,:,:1].squeeze(2)
    # print(type(gt_masks), gt_masks.shape)
    from skimage.segmentation import find_boundaries

    boundaries = find_boundaries(gt_masks, mode='outer')
    boundary_points = np.argwhere(boundaries)
    # print(boundary_points)
    if boundary_points.size == 0:
        bbox = [-1,-1,-1,-1]
    else:
        h0, w0 = boundary_points.min(axis=0)
        h1, w1 = boundary_points.max(axis=0)
        bbox = [w0, h0, w1, h1]
    fin.append({'seg_path':dd['mask'], 'video_path':dd['video_path'], 'bbox':bbox})
    

In [ ]:
len(fin)

In [ ]:
import json
fin2 = []
for dd in fin:
    dd['bbox'] = [int(i) for i in dd['bbox']]
    fin2.append(dd)
with open('davis17_track.json', 'w') as f:
    json.dump(fin2, f, indent = 4)

In [ ]:
fin2[0]